### Used libraries

In [7]:
import pandas as pd
from modules.stopword import stop_words
import re

### Load data from csv

In [8]:
df = pd.read_csv('datasets/job_offer_no_duplicate.csv', delimiter= ',')

print(df.shape)
df.head(3)


(5935, 14)


,Unnamed: 0,url,reference,title,detail_entrerpise,mission,profil,market_category,societe,type_contrat,lieu,societe_slug,limite_date,offers_date
0,0,https://www.portaljob-madagascar.com/emploi/vi...,NaN,electronicien,d'un petit commerce ne il y a plus de cent ans...,L'électronicien industriel est responsable de ...,o solides connaissances en electronique indust...,Ingénierie / industrie / BTP,groupe sipromad,contrat cdi,aucun lieu,groupe sipromad6 electronicien167760,0001-01-01,13/05/2025
1,1,https://www.portaljob-madagascar.com/emploi/vi...,ENG-C-CPP,c/c++ software engineer,"rejoignez astek madagascar, du groupe astek, p...",Participez à des projets de transformation dig...,- formation bac+3 a bac+5 en informatique\n- m...,Informatique / web,astek madagascar,contrat cdd,aucun lieu,astek madagascar c,0001-01-01,28/09/2024
2,2,https://www.portaljob-madagascar.com/emploi/vi...,NaN,manager developpement rh,multisectorielle,Rattaché à la Directrice des Ressources Humain...,"• bac +4/5 en gestion, grh, sociologie, droit,...",Management / RH,inviso group,contrat cdi,antananarivo,fly technologies sarl,0001-01-01,14/02/2025


### Function to clean text

In [14]:
def clean_text(text: str) -> str:
    """
    Clean text to prepare the matching with keyword_mapping.
        - Convert into lowercase
        - delete all punctuations (comma, dot, etc...)
        - Normalise spaces
        - Preserving commonly composed terms like "data scientist", "stagiaire informatique") 
   
    """
    if not isinstance(text, str):
        return ""

    # 1. Convert text to lowercase
    text= text.lower()
    
    # normalise spaces
    text = re.sub(r'\\n+', '\n', text)

    # 2. # Replace underscores, slashes with spaces (to separate words)
    text = re.sub(r"[-_/]", " ", text)

    # 3. # Delete all elements that is not a letter or a digit (with accents), a space, or a number
    text = re.sub(r"[^a-zàâäéèêëîïôöùûüç0-9\s]", " ", text)

    # - replace digits with an empty space
    text = re.sub(r"\d+", " ", text)

    # 4. Normalise multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    # - replacing unecessary terms like 'telma, galaxy', etc with an empty character
    text = re.sub(r"(telma|galaxy|diego|tamatave|axian|antananarivo|mahajanga|toamasina|andraharo|zone|futura|shore|andranomena|immeuble|mdg|batiment ariane|batiment|ariane|tana|antsirabe|fianarantsoa|kube|majunga|tolagnaro|er etage|mdg|mdg campus|campus)", '', text)

    # Removing french articles inside titles
    text = text.split()
    filtered_word = []
    for word in text:
        if word not in stop_words:
            word = filtered_word.append(word)
    text = ' '.join(filtered_word)

    return text


### Function to fix mispelling words

In [10]:
def fix_mispel(text: str) -> str:
    mispell_keyword = {'antananrivo':'antananarivo','en antananarivo':'antananarivo', 'local':'antananarivo', 'aucun lieu':'antananarivo'}
    pattern = re.compile(r'\b(' + '|'.join(mispell_keyword.keys()) + r')\b')
    match = pattern.search(text)
    if match:
        text = text.replace(match.group(), mispell_keyword[match.group()])
    return text


### Clean datasets with clean_text function

In [15]:
df['title_clean'] = df['title'].apply(clean_text)
df['lieu_clean'] = df['lieu'].apply(fix_mispel)
df['mission_clean'] = df['mission'].apply(clean_text)
df['profil_clean'] = df['profil'].apply(clean_text)
df[['title_clean', 'lieu_clean', 'mission_clean','profil_clean']]

,title_clean,lieu_clean,mission_clean,profil_clean
0,electronicien,antananarivo,l électronicien industriel est maintenance rép...,o solides connaissances electronique industrie...
1,c c engineer,antananarivo,participez à projets transformation digitale v...,formation bac bac informatique maitrise techno...
2,developpement rh,antananarivo,rattaché à directrice ressources humaines grou...,bac gestion grh sociologie droit psychologie m...
3,charge commercial,antananarivo,poste basé à tanjombato chargé veille économiq...,titulaire d bac economie marketing sociologie ...
4,projeteur coffrage ferraillage,antananarivo,nous recrutons projeteur génie civil béton arm...,candidat ideal possede bac minimum solide expe...
...,...,...,...,...
5930,boulanger patissier,antananarivo,boulanger pâtissier pain pizza gâteaux cookies...,experience motivee professionnel
5931,securite routiere sinorsr,antananarivo,sécurité routière lieu travail maevana rn,formation bac bac securite prevention risques ...
5932,conformite systeme,antananarivo,comprendre fonctionnement l organisation chaqu...,titulaire d master gestion d diplome d ingenie...
5933,projet digital depot,antananarivo,garantir bonne coordination gestion dépôt gére...,titulaire d master logistique equivalent minim...


### Export data into csv

In [27]:
df = df[['profil','market_category','societe','type_contrat','lieu_clean','limite_date','offers_date','title_clean','mission_clean','profil_clean']]
df.to_csv('datasets/job_offer_cleaned.csv')